Set up Dataset A for global analysis

In [2]:


import json
from pathlib import Path
from collections import Counter

# Dataset A root directory
dataset_a = Path("dataset_A") / "dataset_a_combined"

print("Dataset A path:", dataset_a)
print("Path exists:", dataset_a.exists())

# Find all session directories
sessions = sorted([
    p for p in dataset_a.iterdir()
    if p.is_dir() and p.name.startswith("ses_")
])

print("Number of sessions found:", len(sessions))

# Show first 5 sessions as a quick verification
print("\nFirst 5 sessions:")

for session in sessions[:5]:
    print(session.name)

Dataset A path: dataset_A\dataset_a_combined
Path exists: True
Number of sessions found: 63

First 5 sessions:
ses_20260630-121953-LAPTOP-R36BQBTE
ses_20260630-124826-CHAITANYA0BCF
ses_20260630-125757-LAPTOP-R36BQBTE
ses_20260630-131729-CHAITANYA0BCF
ses_20260630-132737-LAPTOP-R36BQBTE


 Check the basic structure of all 63 sessions

In [3]:


session_structure = []

for session in sessions:

    # Find all event files inside the session
    event_files = list(session.rglob("events.jsonl"))

    # Check for ground-truth files
    gt_file = session / "gt.jsonl"
    gt_manifest = session / "gt_manifest.json"

    session_structure.append({
        "session_id": session.name,
        "event_files": len(event_files),
        "has_gt": gt_file.exists(),
        "has_gt_manifest": gt_manifest.exists()
    })


# Display the first 10 sessions
for row in session_structure[:10]:
    print(
        f"{row['session_id']} | "
        f"event_files={row['event_files']} | "
        f"gt={row['has_gt']} | "
        f"gt_manifest={row['has_gt_manifest']}"
    )


# Overall checks
print("\n" + "=" * 60)

print("Total sessions:", len(session_structure))

print(
    "Sessions with GT:",
    sum(row["has_gt"] for row in session_structure)
)

print(
    "Sessions with GT manifest:",
    sum(row["has_gt_manifest"] for row in session_structure)
)

print(
    "Sessions with at least one event file:",
    sum(row["event_files"] > 0 for row in session_structure)
)

ses_20260630-121953-LAPTOP-R36BQBTE | event_files=2 | gt=True | gt_manifest=True
ses_20260630-124826-CHAITANYA0BCF | event_files=2 | gt=True | gt_manifest=True
ses_20260630-125757-LAPTOP-R36BQBTE | event_files=2 | gt=True | gt_manifest=True
ses_20260630-131729-CHAITANYA0BCF | event_files=2 | gt=True | gt_manifest=True
ses_20260630-132737-LAPTOP-R36BQBTE | event_files=2 | gt=True | gt_manifest=True
ses_20260630-135201-LAPTOP-R36BQBTE | event_files=2 | gt=True | gt_manifest=True
ses_20260630-135451-CHAITANYA0BCF | event_files=2 | gt=True | gt_manifest=True
ses_20260630-145433-JAYESH | event_files=2 | gt=True | gt_manifest=True
ses_20260630-163424-SIDDHIGUPTAB00B | event_files=2 | gt=True | gt_manifest=True
ses_20260630-164418-MSI | event_files=2 | gt=True | gt_manifest=True

Total sessions: 63
Sessions with GT: 63
Sessions with GT manifest: 63
Sessions with at least one event file: 63


Count raw events in every Dataset A session

In [4]:


session_event_counts = []

for session in sessions:

    event_files = list(session.rglob("events.jsonl"))

    total_events = 0

    for event_file in event_files:

        with open(event_file, "r", encoding="utf-8") as f:

            for line in f:
                line = line.strip()

                if line:
                    total_events += 1

    session_event_counts.append({
        "session_id": session.name,
        "event_files": len(event_files),
        "total_events": total_events
    })


# Display first 10 sessions
print("First 10 sessions:\n")

for row in session_event_counts[:10]:
    print(
        f"{row['session_id']} | "
        f"event_files={row['event_files']} | "
        f"events={row['total_events']}"
    )


# Overall statistics
counts = [row["total_events"] for row in session_event_counts]

print("\n" + "=" * 60)

print("Total sessions:", len(counts))
print("Total raw events:", sum(counts))
print("Minimum events in a session:", min(counts))
print("Maximum events in a session:", max(counts))
print("Average events per session:", round(sum(counts) / len(counts), 2))

First 10 sessions:

ses_20260630-121953-LAPTOP-R36BQBTE | event_files=2 | events=2348
ses_20260630-124826-CHAITANYA0BCF | event_files=2 | events=3163
ses_20260630-125757-LAPTOP-R36BQBTE | event_files=2 | events=1799
ses_20260630-131729-CHAITANYA0BCF | event_files=2 | events=2198
ses_20260630-132737-LAPTOP-R36BQBTE | event_files=2 | events=2025
ses_20260630-135201-LAPTOP-R36BQBTE | event_files=2 | events=2020
ses_20260630-135451-CHAITANYA0BCF | event_files=2 | events=3099
ses_20260630-145433-JAYESH | event_files=2 | events=3379
ses_20260630-163424-SIDDHIGUPTAB00B | event_files=2 | events=2483
ses_20260630-164418-MSI | event_files=2 | events=2848

Total sessions: 63
Total raw events: 162768
Minimum events in a session: 945
Maximum events in a session: 3773
Average events per session: 2583.62


Analyze the ground truth across all 63 sessions

In [5]:


gt_summary = []

for session in sessions:

    gt_file = session / "gt.jsonl"

    # Count different GT record types
    record_counts = Counter()

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            event_type = record.get("event")

            if event_type:
                record_counts[event_type] += 1

    gt_summary.append({
        "session_id": session.name,
        "total_gt_records": sum(record_counts.values()),
        "process_started": record_counts["process_started"],
        "process_switched_out": record_counts["process_switched_out"],
        "process_suspended": record_counts["process_suspended"],
        "process_resumed": record_counts["process_resumed"],
        "task_started": record_counts["task_started"],
    })


# Display first 10 sessions
print("First 10 sessions:\n")

for row in gt_summary[:10]:
    print(
        f"{row['session_id']} | "
        f"GT={row['total_gt_records']} | "
        f"starts={row['process_started']} | "
        f"switches={row['process_switched_out']} | "
        f"suspend={row['process_suspended']} | "
        f"resume={row['process_resumed']} | "
        f"tasks={row['task_started']}"
    )


# Dataset-wide totals
print("\n" + "=" * 70)

print("Total sessions:", len(gt_summary))

print(
    "Total GT records:",
    sum(row["total_gt_records"] for row in gt_summary)
)

print(
    "Total process starts:",
    sum(row["process_started"] for row in gt_summary)
)

print(
    "Total process switches:",
    sum(row["process_switched_out"] for row in gt_summary)
)

print(
    "Total suspensions:",
    sum(row["process_suspended"] for row in gt_summary)
)

print(
    "Total resumptions:",
    sum(row["process_resumed"] for row in gt_summary)
)

print(
    "Total task starts:",
    sum(row["task_started"] for row in gt_summary)
)

First 10 sessions:

ses_20260630-121953-LAPTOP-R36BQBTE | GT=175 | starts=31 | switches=26 | suspend=1 | resume=1 | tasks=32
ses_20260630-124826-CHAITANYA0BCF | GT=176 | starts=33 | switches=28 | suspend=1 | resume=1 | tasks=34
ses_20260630-125757-LAPTOP-R36BQBTE | GT=132 | starts=23 | switches=17 | suspend=2 | resume=3 | tasks=26
ses_20260630-131729-CHAITANYA0BCF | GT=136 | starts=26 | switches=19 | suspend=1 | resume=2 | tasks=28
ses_20260630-132737-LAPTOP-R36BQBTE | GT=148 | starts=24 | switches=22 | suspend=2 | resume=3 | tasks=27
ses_20260630-135201-LAPTOP-R36BQBTE | GT=144 | starts=25 | switches=21 | suspend=2 | resume=3 | tasks=28
ses_20260630-135451-CHAITANYA0BCF | GT=186 | starts=32 | switches=27 | suspend=2 | resume=3 | tasks=35
ses_20260630-145433-JAYESH | GT=202 | starts=36 | switches=32 | suspend=1 | resume=1 | tasks=37
ses_20260630-163424-SIDDHIGUPTAB00B | GT=169 | starts=27 | switches=25 | suspend=2 | resume=4 | tasks=31
ses_20260630-164418-MSI | GT=173 | starts=28 | swi

Identify all process types across Dataset A

In [6]:


process_type_counts = Counter()

for session in sessions:

    gt_file = session / "gt.jsonl"

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":

                process_code = record.get("process_code")

                if process_code:
                    process_type_counts[process_code] += 1


print("Number of unique process types:", len(process_type_counts))

print("\nProcess frequencies:\n")

for process, count in sorted(
    process_type_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"Process {process}: {count} starts")

Number of unique process types: 15

Process frequencies:

Process C: 161 starts
Process M: 155 starts
Process H: 147 starts
Process G: 144 starts
Process A: 130 starts
Process N: 129 starts
Process I: 117 starts
Process F: 115 starts
Process D: 114 starts
Process E: 111 starts
Process O: 111 starts
Process J: 102 starts
Process K: 102 starts
Process L: 98 starts
Process B: 83 starts


How widely does each process appear?

In [7]:



process_session_counts = Counter()

# Store which processes appear in each session
session_processes = {}

for session in sessions:

    gt_file = session / "gt.jsonl"

    processes_in_session = set()

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":

                process_code = record.get("process_code")

                if process_code:
                    processes_in_session.add(process_code)

    session_processes[session.name] = processes_in_session

    # Count each process only once per session
    for process in processes_in_session:
        process_session_counts[process] += 1


print("Process coverage across sessions:\n")

for process, count in sorted(
    process_session_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(
        f"Process {process}: "
        f"present in {count}/63 sessions"
    )

Process coverage across sessions:

Process C: present in 43/63 sessions
Process N: present in 43/63 sessions
Process M: present in 42/63 sessions
Process F: present in 41/63 sessions
Process A: present in 40/63 sessions
Process G: present in 39/63 sessions
Process H: present in 39/63 sessions
Process I: present in 39/63 sessions
Process D: present in 38/63 sessions
Process E: present in 37/63 sessions
Process L: present in 37/63 sessions
Process O: present in 37/63 sessions
Process B: present in 35/63 sessions
Process J: present in 34/63 sessions
Process K: present in 32/63 sessions


Session-level process complexity

In [8]:


process_complexity = []

for row in gt_summary:

    process_complexity.append({
        "session_id": row["session_id"],
        "process_starts": row["process_started"],
        "process_switches": row["process_switched_out"],
        "suspensions": row["process_suspended"],
        "resumptions": row["process_resumed"],
        "task_starts": row["task_started"]
    })


# Display first 10 sessions
print("First 10 sessions:\n")

for row in process_complexity[:10]:
    print(
        f"{row['session_id']} | "
        f"starts={row['process_starts']} | "
        f"switches={row['process_switches']} | "
        f"suspend={row['suspensions']} | "
        f"resume={row['resumptions']} | "
        f"tasks={row['task_starts']}"
    )


# Dataset-wide statistics
starts = [row["process_starts"] for row in process_complexity]

print("\n" + "=" * 70)

print("Minimum process starts in a session:", min(starts))
print("Maximum process starts in a session:", max(starts))
print("Average process starts per session:", round(sum(starts) / len(starts), 2))
print("Total process starts:", sum(starts))

First 10 sessions:

ses_20260630-121953-LAPTOP-R36BQBTE | starts=31 | switches=26 | suspend=1 | resume=1 | tasks=32
ses_20260630-124826-CHAITANYA0BCF | starts=33 | switches=28 | suspend=1 | resume=1 | tasks=34
ses_20260630-125757-LAPTOP-R36BQBTE | starts=23 | switches=17 | suspend=2 | resume=3 | tasks=26
ses_20260630-131729-CHAITANYA0BCF | starts=26 | switches=19 | suspend=1 | resume=2 | tasks=28
ses_20260630-132737-LAPTOP-R36BQBTE | starts=24 | switches=22 | suspend=2 | resume=3 | tasks=27
ses_20260630-135201-LAPTOP-R36BQBTE | starts=25 | switches=21 | suspend=2 | resume=3 | tasks=28
ses_20260630-135451-CHAITANYA0BCF | starts=32 | switches=27 | suspend=2 | resume=3 | tasks=35
ses_20260630-145433-JAYESH | starts=36 | switches=32 | suspend=1 | resume=1 | tasks=37
ses_20260630-163424-SIDDHIGUPTAB00B | starts=27 | switches=25 | suspend=2 | resume=4 | tasks=31
ses_20260630-164418-MSI | starts=28 | switches=27 | suspend=2 | resume=5 | tasks=33

Minimum process starts in a session: 23
Maximu

Measure process switching intensity

In [9]:


switching_analysis = []

for row in gt_summary:

    starts = row["process_started"]
    switches = row["process_switched_out"]

    # Ratio of switches to process starts
    switch_ratio = switches / starts if starts > 0 else 0

    switching_analysis.append({
        "session_id": row["session_id"],
        "process_starts": starts,
        "process_switches": switches,
        "switch_ratio": switch_ratio
    })


# Display first 10 sessions
print("First 10 sessions:\n")

for row in switching_analysis[:10]:

    print(
        f"{row['session_id']} | "
        f"starts={row['process_starts']} | "
        f"switches={row['process_switches']} | "
        f"switch_ratio={row['switch_ratio']:.2f}"
    )


# Dataset-wide statistics
ratios = [row["switch_ratio"] for row in switching_analysis]

print("\n" + "=" * 70)

print("Average switch/start ratio:", round(sum(ratios) / len(ratios), 3))
print("Minimum switch/start ratio:", round(min(ratios), 3))
print("Maximum switch/start ratio:", round(max(ratios), 3))

print("\nDataset-wide totals:")
print("Total process starts:", sum(row["process_starts"] for row in switching_analysis))
print("Total process switches:", sum(row["process_switches"] for row in switching_analysis))

First 10 sessions:

ses_20260630-121953-LAPTOP-R36BQBTE | starts=31 | switches=26 | switch_ratio=0.84
ses_20260630-124826-CHAITANYA0BCF | starts=33 | switches=28 | switch_ratio=0.85
ses_20260630-125757-LAPTOP-R36BQBTE | starts=23 | switches=17 | switch_ratio=0.74
ses_20260630-131729-CHAITANYA0BCF | starts=26 | switches=19 | switch_ratio=0.73
ses_20260630-132737-LAPTOP-R36BQBTE | starts=24 | switches=22 | switch_ratio=0.92
ses_20260630-135201-LAPTOP-R36BQBTE | starts=25 | switches=21 | switch_ratio=0.84
ses_20260630-135451-CHAITANYA0BCF | starts=32 | switches=27 | switch_ratio=0.84
ses_20260630-145433-JAYESH | starts=36 | switches=32 | switch_ratio=0.89
ses_20260630-163424-SIDDHIGUPTAB00B | starts=27 | switches=25 | switch_ratio=0.93
ses_20260630-164418-MSI | starts=28 | switches=27 | switch_ratio=0.96

Average switch/start ratio: 0.874
Minimum switch/start ratio: 0.714
Maximum switch/start ratio: 1.0

Dataset-wide totals:
Total process starts: 1819
Total process switches: 1590


Analyze suspensions and resumptions

In [10]:


suspension_analysis = []

for row in gt_summary:

    suspension_analysis.append({
        "session_id": row["session_id"],
        "suspensions": row["process_suspended"],
        "resumptions": row["process_resumed"]
    })


print("First 10 sessions:\n")

for row in suspension_analysis[:10]:

    print(
        f"{row['session_id']} | "
        f"suspensions={row['suspensions']} | "
        f"resumptions={row['resumptions']}"
    )


print("\n" + "=" * 70)

print(
    "Sessions with at least one suspension:",
    sum(row["suspensions"] > 0 for row in suspension_analysis)
)

print(
    "Sessions with at least one resumption:",
    sum(row["resumptions"] > 0 for row in suspension_analysis)
)

print(
    "Total suspensions:",
    sum(row["suspensions"] for row in suspension_analysis)
)

print(
    "Total resumptions:",
    sum(row["resumptions"] for row in suspension_analysis)
)

First 10 sessions:

ses_20260630-121953-LAPTOP-R36BQBTE | suspensions=1 | resumptions=1
ses_20260630-124826-CHAITANYA0BCF | suspensions=1 | resumptions=1
ses_20260630-125757-LAPTOP-R36BQBTE | suspensions=2 | resumptions=3
ses_20260630-131729-CHAITANYA0BCF | suspensions=1 | resumptions=2
ses_20260630-132737-LAPTOP-R36BQBTE | suspensions=2 | resumptions=3
ses_20260630-135201-LAPTOP-R36BQBTE | suspensions=2 | resumptions=3
ses_20260630-135451-CHAITANYA0BCF | suspensions=2 | resumptions=3
ses_20260630-145433-JAYESH | suspensions=1 | resumptions=1
ses_20260630-163424-SIDDHIGUPTAB00B | suspensions=2 | resumptions=4
ses_20260630-164418-MSI | suspensions=2 | resumptions=5

Sessions with at least one suspension: 63
Sessions with at least one resumption: 62
Total suspensions: 99
Total resumptions: 190


Global process transition counts

In [11]:


transition_counts = Counter()

for session in sessions:

    gt_file = session / "gt.jsonl"

    # Store process switches in chronological order
    process_switches = []

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_switched_out":

                from_process = record.get("from")
                to_process = record.get("to")

                if from_process and to_process:
                    process_switches.append(
                        (from_process, to_process)
                    )

    # Count transitions for this session
    for transition in process_switches:
        transition_counts[transition] += 1


print("Global process transitions:\n")

for (from_process, to_process), count in sorted(
    transition_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(
        f"{from_process} -> {to_process}: {count}"
    )


print("\n" + "=" * 70)

print(
    "Number of unique transition types:",
    len(transition_counts)
)

print(
    "Total recorded transitions:",
    sum(transition_counts.values())
)

Global process transitions:

M -> C: 18
G -> H: 17
G -> J: 17
A -> I: 15
C -> M: 15
F -> A: 15
M -> A: 15
A -> D: 13
A -> F: 13
B -> A: 13
C -> A: 13
F -> C: 13
H -> K: 13
K -> F: 13
M -> N: 13
N -> M: 13
A -> B: 12
A -> C: 12
A -> E: 12
C -> B: 12
C -> F: 12
C -> O: 12
E -> F: 12
F -> B: 12
F -> M: 12
G -> K: 12
G -> L: 12
H -> L: 12
I -> L: 12
L -> I: 12
N -> F: 12
C -> G: 11
C -> I: 11
C -> N: 11
F -> E: 11
G -> M: 11
H -> G: 11
K -> G: 11
L -> G: 11
M -> F: 11
M -> G: 11
A -> M: 10
C -> E: 10
D -> F: 10
E -> M: 10
F -> L: 10
H -> B: 10
H -> J: 10
H -> M: 10
H -> N: 10
I -> C: 10
I -> H: 10
I -> N: 10
K -> H: 10
K -> L: 10
L -> H: 10
M -> L: 10
O -> E: 10
O -> M: 10
A -> G: 9
A -> L: 9
B -> F: 9
C -> H: 9
D -> A: 9
D -> B: 9
D -> C: 9
E -> C: 9
I -> A: 9
I -> F: 9
M -> D: 9
M -> K: 9
N -> A: 9
N -> B: 9
N -> C: 9
N -> D: 9
A -> K: 8
B -> D: 8
D -> K: 8
D -> N: 8
D -> O: 8
E -> N: 8
F -> D: 8
F -> I: 8
G -> A: 8
G -> B: 8
H -> A: 8
I -> B: 8
J -> M: 8
K -> M: 8
K -> N: 8
L -> M: 8
M 

How concentrated are the transitions?

In [12]:


transition_values = list(transition_counts.values())

print("Transition frequency statistics:\n")

print("Unique transition types:", len(transition_values))
print("Total transitions:", sum(transition_values))
print("Minimum frequency:", min(transition_values))
print("Maximum frequency:", max(transition_values))
print(
    "Average frequency per transition:",
    round(sum(transition_values) / len(transition_values), 2)
)

# Show the 20 most frequent transitions
print("\nTop 20 most frequent transitions:\n")

for (from_process, to_process), count in sorted(
    transition_counts.items(),
    key=lambda x: (-x[1], x[0])
)[:20]:

    print(
        f"{from_process} -> {to_process}: {count}"
    )

Transition frequency statistics:

Unique transition types: 210
Total transitions: 1590
Minimum frequency: 1
Maximum frequency: 18
Average frequency per transition: 7.57

Top 20 most frequent transitions:

M -> C: 18
G -> H: 17
G -> J: 17
A -> I: 15
C -> M: 15
F -> A: 15
M -> A: 15
A -> D: 13
A -> F: 13
B -> A: 13
C -> A: 13
F -> C: 13
H -> K: 13
K -> F: 13
M -> N: 13
N -> M: 13
A -> B: 12
A -> C: 12
A -> E: 12
C -> B: 12


Process duration analysis

In [13]:


from datetime import datetime

process_start_gaps = []

for session in sessions:

    gt_file = session / "gt.jsonl"

    process_starts_session = []

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":

                timestamp = record.get("ts_utc")

                if timestamp:
                    process_starts_session.append(
                        datetime.fromisoformat(timestamp)
                    )

    # Sort chronologically
    process_starts_session.sort()

    # Calculate gaps between consecutive process starts
    for i in range(1, len(process_starts_session)):

        gap = (
            process_starts_session[i]
            - process_starts_session[i - 1]
        ).total_seconds()

        process_start_gaps.append(gap)


print("Process-start temporal spacing:\n")

print(
    "Number of gaps:",
    len(process_start_gaps)
)

print(
    "Minimum gap (seconds):",
    round(min(process_start_gaps), 3)
)

print(
    "Maximum gap (seconds):",
    round(max(process_start_gaps), 3)
)

print(
    "Average gap (seconds):",
    round(
        sum(process_start_gaps) / len(process_start_gaps),
        3
    )
)

# Show a few smallest and largest gaps
print("\nSmallest 10 gaps:")

for gap in sorted(process_start_gaps)[:10]:
    print(f"{gap:.3f} sec")

print("\nLargest 10 gaps:")

for gap in sorted(process_start_gaps)[-10:]:
    print(f"{gap:.3f} sec")

Process-start temporal spacing:

Number of gaps: 1756
Minimum gap (seconds): 16.91
Maximum gap (seconds): 1144.837
Average gap (seconds): 46.656

Smallest 10 gaps:
16.910 sec
17.476 sec
17.738 sec
18.111 sec
18.142 sec
18.246 sec
18.295 sec
18.384 sec
18.498 sec
18.504 sec

Largest 10 gaps:
200.692 sec
208.626 sec
235.684 sec
248.966 sec
290.234 sec
290.234 sec
302.334 sec
485.127 sec
749.107 sec
1144.837 sec


Process - Application relationship

In [14]:
# Cell 13: Analyze process-to-application relationships

process_apps = {}

for session in sessions:

    gt_file = session / "gt.jsonl"

    # Load process start times
    process_starts = []

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":

                process = record.get("process_code")
                ts = record.get("ts_utc")

                if process and ts:
                    process_starts.append(
                        (datetime.fromisoformat(ts), process)
                    )

    # Load raw events
    events = []

    for event_file in session.rglob("events.jsonl"):

        with open(event_file, "r", encoding="utf-8") as f:

            for line in f:
                line = line.strip()

                if line:
                    event = json.loads(line)

                    if event.get("timestamp_iso"):
                        events.append(event)

    # Sort events chronologically
    events.sort(
        key=lambda x: x.get("timestamp_ms", 0)
    )

    # Associate nearby events with the most recent process start
    for start_time, process in process_starts:

        if process not in process_apps:
            process_apps[process] = Counter()

        for event in events:

            event_time = event.get("timestamp_iso")

            if not event_time:
                continue

            try:
                event_time = datetime.fromisoformat(
                    event_time.replace("Z", "+00:00")
                )
            except ValueError:
                continue

            # Consider events from process start
            # until 10 seconds after it
            time_diff = (
                event_time - start_time
            ).total_seconds()

            if 0 <= time_diff <= 10:

                app = (
                    event.get("context", {})
                    or {}
                ).get("active_app")

                if isinstance(app, dict):
                    app_name = app.get("app_name")

                    if app_name:
                        process_apps[process][app_name] += 1


print("Top applications associated with each process:\n")

for process in sorted(process_apps):

    print(f"\nProcess {process}:")

    for app, count in process_apps[process].most_common(5):
        print(f"  {app}: {count}")

Top applications associated with each process:


Process A:
  Google Chrome: 4579
  Microsoft Excel: 131
  olk: 34
  WindowsTerminal: 31
  procmine-desktop-agent: 29

Process B:
  Google Chrome: 3068
  Microsoft PowerPoint: 34
  WindowsTerminal: 34
  Microsoft Excel: 32
  olk: 28

Process C:
  Google Chrome: 3578
  Microsoft Word: 208
  Windows Explorer: 97
  Notepad: 59
  Microsoft Excel: 32

Process D:
  Google Chrome: 2337
  Notepad: 518
  Windows Explorer: 94
  WindowsTerminal: 34
  Microsoft Excel: 19

Process E:
  Google Chrome: 3701
  Notepad: 89
  Windows Explorer: 63
  Microsoft Excel: 6

Process F:
  Google Chrome: 4232
  WindowsTerminal: 156
  Microsoft Excel: 116
  Windows Explorer: 59
  Notepad: 5

Process G:
  Google Chrome: 5021
  Notepad: 168
  WindowsTerminal: 96
  Windows Explorer: 34
  Microsoft Excel: 29

Process H:
  Google Chrome: 5053
  WindowsTerminal: 133
  Notepad: 62
  Microsoft Excel: 35
  Windows Explorer: 25

Process I:
  Google Chrome: 2506
  Microsoft Po

Event-type patterns by process

In [15]:


from collections import Counter, defaultdict
from datetime import datetime

# Store event-type counts for each process
process_event_types = defaultdict(Counter)

# Store number of process instances analyzed
process_instance_counts = Counter()

for session in sessions:

    gt_file = session / "gt.jsonl"

    # -----------------------------
    # Load process start timestamps
    # -----------------------------
    process_starts = []

    with open(gt_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":

                process = record.get("process_code")
                timestamp = record.get("ts_utc")

                if process and timestamp:
                    process_starts.append(
                        (
                            datetime.fromisoformat(timestamp),
                            process
                        )
                    )

    # -----------------------------
    # Load all raw events
    # -----------------------------
    events = []

    for event_file in session.rglob("events.jsonl"):

        with open(event_file, "r", encoding="utf-8") as f:

            for line in f:
                line = line.strip()

                if not line:
                    continue

                event = json.loads(line)

                if event.get("timestamp_iso"):
                    events.append(event)

    # Sort raw events chronologically
    events.sort(
        key=lambda x: x.get("timestamp_ms", 0)
    )

    # -----------------------------
    # Analyze events after each
    # process start
    # -----------------------------
    for start_time, process in process_starts:

        process_instance_counts[process] += 1

        for event in events:

            timestamp = event.get("timestamp_iso")

            if not timestamp:
                continue

            try:
                event_time = datetime.fromisoformat(
                    timestamp.replace("Z", "+00:00")
                )
            except ValueError:
                continue

            time_diff = (
                event_time - start_time
            ).total_seconds()

            # 10-second exploratory window
            if 0 <= time_diff <= 10:

                event_type = event.get("event_type")

                if event_type:
                    process_event_types[process][event_type] += 1


# ------------------------------------
# Display event-type patterns
# ------------------------------------

print("Event-type patterns by process:\n")

for process in sorted(process_event_types):

    print(f"\nProcess {process}")
    print(
        f"Instances analyzed: "
        f"{process_instance_counts[process]}"
    )

    total = sum(
        process_event_types[process].values()
    )

    for event_type, count in process_event_types[process].most_common():

        percentage = (
            count / total * 100
            if total > 0
            else 0
        )

        print(
            f"  {event_type}: "
            f"{count} ({percentage:.1f}%)"
        )

Event-type patterns by process:


Process A
Instances analyzed: 130
  app_switch: 2159 (44.8%)
  screenshot_smart: 930 (19.3%)
  keystroke: 914 (19.0%)
  shortcut: 359 (7.4%)
  mouse_click: 130 (2.7%)
  browser_click: 118 (2.4%)
  clipboard_change: 91 (1.9%)
  browser_navigation: 82 (1.7%)
  mouse_scroll: 37 (0.8%)
  upload_completed: 1 (0.0%)

Process B
Instances analyzed: 83
  app_switch: 1469 (45.6%)
  keystroke: 620 (19.2%)
  screenshot_smart: 605 (18.8%)
  shortcut: 236 (7.3%)
  mouse_click: 80 (2.5%)
  browser_click: 73 (2.3%)
  clipboard_change: 60 (1.9%)
  browser_navigation: 55 (1.7%)
  mouse_scroll: 20 (0.6%)
  upload_completed: 2 (0.1%)
  upload_started: 1 (0.0%)

Process C
Instances analyzed: 161
  app_switch: 2532 (63.3%)
  screenshot_smart: 798 (20.0%)
  mouse_click: 155 (3.9%)
  browser_click: 136 (3.4%)
  keystroke: 132 (3.3%)
  browser_navigation: 99 (2.5%)
  shortcut: 60 (1.5%)
  mouse_scroll: 58 (1.5%)
  window_state_change: 16 (0.4%)
  window_title_change: 12 (0.3%)

FINAL GLOBAL AUDIT SUMMARY

In [17]:


print("=" * 70)
print("DATASET A — GLOBAL AUDIT SUMMARY")
print("=" * 70)


# DATASET STRUCTURE

print("\nDATASET STRUCTURE")
print("-----------------")

print(f"Total sessions: {len(sessions)}")

print(
    f"Sessions with GT: "
    f"{sum(row['has_gt'] for row in session_structure)}/63"
)

print(
    f"Sessions with GT manifest: "
    f"{sum(row['has_gt_manifest'] for row in session_structure)}/63"
)

print(
    f"Sessions with event files: "
    f"{sum(row['event_files'] > 0 for row in session_structure)}/63"
)



# RAW EVENTS

print("\nRAW EVENTS")
print("----------")

total_raw_events = sum(counts)

print(f"Total raw events: {total_raw_events:,}")
print(f"Minimum events/session: {min(counts):,}")
print(f"Maximum events/session: {max(counts):,}")
print(f"Average events/session: {total_raw_events / len(counts):.2f}")


# GROUND TRUTH

print("\nGROUND TRUTH")
print("------------")

total_gt = sum(row["total_gt_records"] for row in gt_summary)
total_starts = sum(row["process_started"] for row in gt_summary)
total_switches = sum(row["process_switched_out"] for row in gt_summary)
total_suspensions = sum(row["process_suspended"] for row in gt_summary)
total_resumptions = sum(row["process_resumed"] for row in gt_summary)
total_tasks = sum(row["task_started"] for row in gt_summary)

print(f"Total GT records: {total_gt:,}")
print(f"Process starts: {total_starts:,}")
print(f"Process switches: {total_switches:,}")
print(f"Suspensions: {total_suspensions:,}")
print(f"Resumptions: {total_resumptions:,}")
print(f"Task starts: {total_tasks:,}")


# PROCESS TYPES

print("\nPROCESS TYPES")
print("-------------")

print(
    f"Unique process types: "
    f"{len(process_type_counts)}"
)

print(
    "Process types:",
    ", ".join(sorted(process_type_counts))
)



# PROCESS COVERAGE

print("\nPROCESS COVERAGE")
print("----------------")

for process, count in sorted(
    process_session_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"{process}: {count}/63 sessions")



# PROCESS COMPLEXITY

print("\nPROCESS COMPLEXITY")
print("------------------")

session_process_starts = [
    row["process_started"]
    for row in gt_summary
]

print(
    f"Process starts/session: "
    f"{min(session_process_starts)} – "
    f"{max(session_process_starts)}"
)

print(
    f"Average process starts/session: "
    f"{sum(session_process_starts) / len(session_process_starts):.2f}"
)



# PROCESS SWITCHING

print("\nPROCESS SWITCHING")
print("-----------------")

total_transitions = sum(transition_counts.values())

print(
    f"Unique transition types: "
    f"{len(transition_counts)}"
)

print(
    f"Total transitions: "
    f"{total_transitions}"
)

print(
    f"Switch/start ratio: "
    f"{total_switches / total_starts:.3f}"
)



# SUSPENSION / RESUMPTION

print("\nSUSPENSION / RESUMPTION")
print("-----------------------")

sessions_with_suspension = sum(
    row["suspensions"] > 0
    for row in suspension_analysis
)

sessions_with_resumption = sum(
    row["resumptions"] > 0
    for row in suspension_analysis
)

print(
    f"Sessions with suspension: "
    f"{sessions_with_suspension}/63"
)

print(
    f"Sessions with resumption: "
    f"{sessions_with_resumption}/63"
)



# TEMPORAL PATTERNS

print("\nTEMPORAL PATTERNS")
print("-----------------")

print(
    f"Minimum process-start gap: "
    f"{min(process_start_gaps):.2f} sec"
)

print(
    f"Average process-start gap: "
    f"{sum(process_start_gaps) / len(process_start_gaps):.2f} sec"
)

print(
    f"Maximum process-start gap: "
    f"{max(process_start_gaps):.2f} sec"
)



# KEY CONCLUSIONS

print("\nKEY CONCLUSIONS")
print("---------------")

print("1. Dataset A contains substantial process switching and interleaving.")
print("2. The same process type can occur in multiple cases.")
print("3. Process transitions are not sufficient as the sole segmentation signal.")
print("4. Application identity alone is not sufficient for process identification.")
print("5. Event-type composition alone is not sufficient for process identification.")
print("6. Process-start timing is highly variable.")
print("7. Suspension/resumption indicates non-contiguous process activity.")
print("8. A segmentation approach will require multiple event-level signals.")

print("\n" + "=" * 70)
print("GLOBAL AUDIT COMPLETE")
print("=" * 70)

DATASET A — GLOBAL AUDIT SUMMARY

DATASET STRUCTURE
-----------------
Total sessions: 63
Sessions with GT: 63/63
Sessions with GT manifest: 63/63
Sessions with event files: 63/63

RAW EVENTS
----------
Total raw events: 162,768
Minimum events/session: 945
Maximum events/session: 3,773
Average events/session: 2583.62

GROUND TRUTH
------------
Total GT records: 10,609
Process starts: 1,819
Process switches: 1,590
Suspensions: 99
Resumptions: 190
Task starts: 2,009

PROCESS TYPES
-------------
Unique process types: 15
Process types: A, B, C, D, E, F, G, H, I, J, K, L, M, N, O

PROCESS COVERAGE
----------------
C: 43/63 sessions
N: 43/63 sessions
M: 42/63 sessions
F: 41/63 sessions
A: 40/63 sessions
G: 39/63 sessions
H: 39/63 sessions
I: 39/63 sessions
D: 38/63 sessions
E: 37/63 sessions
L: 37/63 sessions
O: 37/63 sessions
B: 35/63 sessions
J: 34/63 sessions
K: 32/63 sessions

PROCESS COMPLEXITY
------------------
Process starts/session: 23 – 43
Average process starts/session: 28.87

PROC